# Annotated E(n)-Equivariant Graph Neural Network

> A practical EGNN walkthrough built around QM9, with an eye toward later
> extensions into geometry-aware generative modelling.

![EGNN diagram](./img/header.png)

## So what are the overall goals?

The goal here is to build a solid **E(n)-Equivariant Graph Neural Network** for molecules without turning the whole exercise into a black box. I wanted a notebook that is simple enough to run end-to-end, but still detailed enough that each design choice is motivated rather than hidden away in library code.

As a first step, I use **QM9** and train an **EGNN** to predict a molecular property from 3D structure. Once that pipeline is working cleanly, the same backbone can be adapted to larger datasets, conformer-based tasks, or eventually **equivariant diffusion / flow-matching**.

## What is QM9?

**QM9** contains roughly 130K small molecules and is built into `torch_geometric`. It is a useful starting point because it gives us small graphs, atom and bond information, and 3D coordinates without too much preprocessing overhead.

That makes it a good starting point for setting up and debugging the core pieces of the pipeline:

- batching;
- graph construction;
- coordinate handling;
- EGNN forward pass;
- training loop

Once those pieces work cleanly, it becomes much easier to move on to more complex tasks.

### Some notes before we start
- This is not meant to be a full introduction to graph neural networks or equivariance. For background, the original EGNN paper is a good starting point: https://proceedings.mlr.press/v139/satorras21a.html. For a gentler introduction to GNNs, I also recommend: https://distill.pub/2021/gnn-intro/
- The level of annotation here is intentionally high. The audience I had in mind was someone learning the ideas and wanting to see how they show up in code.
- The notebook is written as a worked tutorial rather than a benchmark script, so clarity takes priority over compactness.

In [ ]:
'''A dedicated environment avoid dependency hell.'''

# Example environment setup (from the terminal):
# conda create -n egnn python=3.11 -y
# conda activate egnn
# python -m pip install --upgrade pip

# Optional installs in a fresh environment:
# !pip install torch torchvision torchaudio
# !pip install torch-geometric
# !pip install rdkit
# !pip install tqdm numpy pandas matplotlib

In [ ]:
import os; import math; import random
from dataclasses import dataclass
from typing import Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm.auto import tqdm

from torch_geometric.datasets import QM9 
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_add_pool

## Setting up the dataset
As discussed above, we use **QM9** as a **baseline**.
My goal is to train a model to predict the **HOMO-LUMO gap** of small molecules.

---

__Aside__ [Skip if you do not need the chemistry context]

HOMO stands for the 'highest occupied molecular orbital'. This is the energy level of the most energetic orbital that currently holds electrons. Simply put, the most likely source of electrons for any reaction. 

LUMO stands for the 'lowest unoccupied molecular orbital'. This is the energy level of the least energetic orbital that is completely empty. It is the likely spot for accepting electrons for any reaction. 

The energy gap between the HOMO and the LUMO, simply defined as
$$
    E_{gap} = E_{LUMO} - E_{HOMO}
$$
becomes an important metric to assess the stability or reactivity of a molecule. A small gap implies the molecule is more reactive, as even small amounts of energy can transfer an electron between orbitals. A large gap usually implies higher stability. In practice, this matters for applications such as photocatalysis, where light can excite electrons and induce a chemical reaction.  

p.s., now you and I know the same amount of chemistry

---

In the PyTorch Geometric QM9 dataset, the target index `7` corresponds to the HOMO-LUMO gap. 

From a model-training point of view, this is a good starter target because:
- it is chemically meaningful;
- it is scalar; AND
- it lets us verify the whole geometry-aware learning pipeline

In [ ]:
'''Model configuration file; this includes hyperparameters and train/test/val splits
To change any model hyperparameters during fine-tuning, we will simply change them in the config.'''

@dataclass
class Config:
    root: str = "data/QM9"
    homo_lumo_idx: int = 7              # index of HOMO-LUMO gap in QM9
    batch_size: int = 96                # can be larger if trained on a GPU
    hidden_dim: int = 128    
    message_dim: int = 128
    num_layers: int = 6      
    dropout: float = 0.0      
    lr: float = 1e-3                    # learning rate of training                     
    weight_decay: float = 1e-10
    epochs: int = 50                    
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    seed: int = 42
    coord_step_size: float = 0.1        # to stabilise training (discussed later)
    eps: float = 1e-8
    train_split: float = 0.8
    val_split: float = 0.1              # test = 1 - train - val

cfg = Config(); cfg

In [ ]:
'''Setting a seed helps better reproduce performance and behavior'''
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(cfg.seed)

#### Loading QM9 dataset
PyTorch-geometric QM9 object already gives us graph-structured molecules with 3D coordinates.
Each graph contains:
- `x`: node features
- `pos`: atomic coordinates
- `edge_index`: graph connectivity
- `edge_attr`: edge features
- `y`: molecular targets

---

**Important note:** When I tried to process QM9 directly through PyG, I ran into errors from a small number of invalid molecules. This may be dependency/version related, but it also gives us a useful excuse to build the graph objects ourselves. On the bright side, this lets us peek under the hood and understand how molecules become graphs.

Important life lesson --> always look on the bright side; unless you're already on the bright side.

In [ ]:
'''
The standard command to download and process QM9 is:
dataset = QM9(root = cfg.root)

If this works in your environment, you can use it directly. If it fails during
processing, the custom preprocessing code below is a useful fallback.
'''

# --- Some extra dependencies ---
import pandas as pd
from pathlib import Path
from rdkit import Chem
from torch_geometric.data import Data 
from torch_geometric.utils import one_hot

'''
This is slightly messy, but first run:
dataset = QM9(root = cfg.root)

This will download the raw data and then during processing, give an error.
Once downloaded, we run the processing script shown below.
'''

load_path = r'data/QM9/raw/gdb9.sdf' 
save_path = r'data/QM9/processed/qm9_processed.pt'
csv_path = r'data/QM9/raw/gdb9.sdf.csv'

'''
PyG QM9 stores 19 properties for each molecule. Here we process all of them,
so that the same dataset can later be used to predict any available property
of interest.
'''

TARGET_KEYS = ["A", "B", "C", "mu", "alpha", "homo", "lumo", "gap", "r2", 
               "zpve", "u0", "u298", "h298", "g298", "cv", "u0_atom", "u298_atom", 
               "h298_atom", "g298_atom"]

ATOM_TYPES = ["H", "C", "N", "O", "F"] # For context: see note below
BOND_TYPES = [Chem.rdchem.BondType.SINGLE, Chem.rdchem.BondType.DOUBLE,
              Chem.rdchem.BondType.TRIPLE, Chem.rdchem.BondType.AROMATIC]

def atom_features(atom, num_hs: int):
    '''Generating a set of atom features (for every atom in the molecule).
    In graph terms --> this translates to node features.
    This returns a vector which contains:
    - one hot encoding of atom type
    - atomic number
    - aromaticity (0: no/ 1: yes)
    - one hot encoding of hybridization (sp/sp2/sp3)
    - number of connected hydrogens
    '''
    hybridization = atom.GetHybridization()
    return [
        1.0 if atom.GetSymbol() == atom_type else 0.0 for atom_type in ATOM_TYPES
    ] + [float(atom.GetAtomicNum()),
        float(atom.GetIsAromatic()),
        float(hybridization == Chem.rdchem.HybridizationType.SP),
        float(hybridization == Chem.rdchem.HybridizationType.SP2),
        float(hybridization == Chem.rdchem.HybridizationType.SP3),
        float(num_hs),]

def mol_to_graph(mol, y):
    '''
    We need to take the molecules in the QM9 and convert it into a graph representation
    for downstream processing. Most of the heavy lifting is done using rdKit.
    '''
    conf = mol.GetConformer() # See note below for more context.
    pos = torch.tensor(conf.GetPositions(), dtype = torch.float) # store atomic positions in 3d

    z = torch.tensor([atom.GetAtomicNum() for atom in mol.GetAtoms()], dtype = torch.long) # store atom type
    num_atoms = z.size(0) 

    rows, cols, edge_types = [], [], []
    '''VERY IMPORTANT:
    `rows` and `cols` are the two lists that will later become `edge_index`'''

    for bond in mol.GetBonds():
        '''Iterate through all bonds present in the molecule'''
        start, end = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx() # Which atoms (index) the bond is between
        bond_type = BOND_TYPES.index(bond.GetBondType()) # what is the bond type

        rows += [start, end] # source nodes for the two directed bond edges
        cols += [end, start] # matching destination nodes
        edge_types += [bond_type, bond_type]

    if len(rows) == 0:
        edge_index = torch.empty((2, 0), dtype = torch.long)
        edge_attr = torch.empty((0, 4), dtype = torch.float)
    else:
        edge_index = torch.tensor([rows, cols], dtype = torch.long)
        edge_type = torch.tensor(edge_types, dtype = torch.long)
        edge_attr = one_hot(edge_type, num_classes = len(BOND_TYPES)).to(torch.float)
        '''Generating a simple set of bond features (for every bond in the molecule).
        In graph terms --> this translates to edge attributes.
        This returns a vector which contains:
        - one hot encoding of bond type (single/double/triple/aromatic)
        '''

        '''Now we need to sort edges to arrange them in canonical order
        (0, 0), (0, 1), ... , (0, n), (1, 0), ... (n, n)
        '''
        perm = (edge_index[0] * num_atoms + edge_index[1]).argsort() # sort edges in canonical order
        edge_index = edge_index[:, perm]
        edge_attr = edge_attr[perm]

    hs = (z == 1).to(torch.float) # 1: atom = H, 0: otherwise
    num_hs = torch.zeros(num_atoms, dtype = torch.float) 
    if edge_index.numel() > 0:
        row, col = edge_index # row: edge start, col: edge end
        num_hs.index_add_(0, col, hs[row]) # for each edge, add hs[row] to hs[col]
        # num_hs[i] is the number of H atoms bonded to atom i

    x = torch.tensor(
        [atom_features(atom, int(num_hs[i].item())) for i, atom in enumerate(mol.GetAtoms())],
        dtype=torch.float,
    )

    data = Data(x = x, z = z, pos = pos, edge_index = edge_index, edge_attr = edge_attr, y = y)
    return data


def preprocess(path: str, csv_path: str):
    raw_sdf = Path(path)
    suppl = Chem.SDMolSupplier(str(raw_sdf), removeHs = False)
    df = pd.read_csv(csv_path)
    data_list = []

    for idx, mol in enumerate(tqdm(suppl, desc = 'Processing QM9 (...)')):
        if mol is None or idx >= len(df): continue
        
        try:
            row = df.iloc[idx]
            if row[TARGET_KEYS].isna().any(): continue

            vals = row[TARGET_KEYS].to_numpy(dtype = float)
            y = torch.tensor(vals, dtype = torch.float).unsqueeze(0)

            data = mol_to_graph(mol, y)
            data_list.append(data)
        
        except (ValueError, RuntimeError, IndexError) as exc:
            print(f"Skipping molecule at index {idx}: {type(exc).__name__}: {exc}")
            continue
    
    return data_list

data_list = preprocess(load_path, csv_path)
print(f"Kept {len(data_list)} molecules post-processing")
torch.save(data_list, save_path)

#### What exactly happened in that preprocessing block?

Let us go through the data processing step by step. QM9 is a collection of molecules, but the model expects graph objects with tensors attached to them. The job of the preprocessing code is to convert each molecule into that graph representation.

---

First we have 
```python
def atom_features(atom, num_hs: int):
```
This function takes as input an atom (of the molecule) and the number of hydrogens connected to it and returns atom_features vector as described in the schematic below.

![atom_features](./img/af.png)

This will serve as the node features for the **egnn**

---

Next we have
```python
def mol_to_graph(mol, y)
```
This function takes a molecule (`rdkit.Mol` format) and the target to predict (`y`), and does most of the heavy lifting.
- We start by embedding this molecule in 3d ```mol.GetConformer()```
- Following this, we extract cartesian coordinates (x, y, z) into a tensor using ```conf.GetPositions()```
- A little note here; computational chemistry people may not be entirely happy with this step. Ideally we would want to generate multiple conformers and select the one with the lowest energy. We skip this step in the interest of saving some compute time.


In PyG, `edge_index` has shape [2, num_edges]. The first row stores source nodes, 
and the second row stores destination / target nodes.

So if at some position k we have:
```python 
rows[k] = i
cols[k] = j
```

then this means there is a directed edge i -> j.

Example:
```python
rows = [0, 1, 1, 2]
cols = [1, 0, 2, 1]
```
corresponds to the directed edges: $$0 \rightarrow 1, 1 \rightarrow 0, 1 \rightarrow 2, 2 \rightarrow 1$$

A chemical bond is undirected, but message-passing GNNs usually store it as
two directed edges so that information can flow both ways. That is why every
bond contributes BOTH: $$start \rightarrow end \ \ AND \ \ end \rightarrow start$$

We then stack these two lists as: 
```python
edge_index = torch.tensor([rows, cols])
```
which is the standard PyG edge-list representation.

Following this, we iterate through all bonds in the molecule, collect the starting and ending index and extract the bond type (SINGLE, DOUBLE, TRIPLE OR AROMATIC). We start compiling the rows (start, end) and cols (end, start) along with the bond type to build a complete edge mapping. Lastly, we one-hot encode the bond_type. 

![edge_features](./img/ef.png)

---

The final compiled dataset contains:
- data.x : atom (node) features
- data.z : atomic numbers
- data.pos : 3d coordinates for each atom 
- data.edge_index : edge list (row --> col and col --> row)
- data.edge_attr : bond (edge) features
- data.y : target (HOMO-LUMO gap)

### Let's convert this to a custom dataset

Now we compile everything together into a neat molecule dataset. This will be compatible with model training that is coming up soon.

In [ ]:
from torch.utils.data import Dataset
class MoleculeDataset(Dataset):
    def __init__(self, processed_file):
        self.qm9 = torch.load(processed_file, weights_only = False, map_location = 'cpu')
    def __len__(self): return len(self.qm9)
    def __getitem__(self, index): return self.qm9[index]

    @property
    def num_features(self):
        sample = self.qm9[0]
        return sample.x.size(-1) if hasattr(sample, "x") and sample.x is not None else 0

    @property
    def num_node_features(self):
        return self.num_features

    @property
    def num_edge_features(self):
        sample = self.qm9[0]
        return sample.edge_attr.size(-1) if hasattr(sample, "edge_attr") and sample.edge_attr is not None else 0

processed_path = 'data/QM9/processed/qm9_processed.pt' # change this if you move the processed file.
dataset = MoleculeDataset(processed_path)

debug_dataset = False
if debug_dataset:
    '''Let's do a quick test to see if the dataloader works.'''
    data = dataset[42]
    print("Data object keys:", data.keys)
    print(f"x: {data.x}; x shape {data.x.shape}")
    print(f"z: {data.z}; z shape {data.z.shape}")
    print(f"y: {data.y}; y shape {data.y.shape}")
    print(f"pos: {data.pos}; pos shape {data.pos.shape}")
    print('Test passed')

else:
    print(dataset)
    print("Number of molecules:", len(dataset))
    print("Node feature dimension:", dataset.num_features)
    print("Example edge feature dimension:", dataset.num_edge_features)
    print("Number of targets:", dataset[0].y.shape[-1])
 

Next we generate a train/val/test split; standard 80/10/10

In [ ]:
from torch.utils.data import random_split
from typing import List
class BuildLoader():
    def __init__(self, datapath, split: List[float]):
        self.dataset = MoleculeDataset(datapath)
        self.len = len(self.dataset)
        self.train_split = split[0]; self.val_split = split[1]
        self.n_train = int(self.train_split * self.len)
        self.n_val = int(self.val_split * self.len)
        self.n_test = self.len - self.n_train - self.n_val 

    def generate_split(self, seed):
        torch.manual_seed(seed)
        train, val, test = random_split(self.dataset, [self.n_train, self.n_val, self.n_test])
        return train, val, test
        
    def load_split(self, seed: int, batch: int):
        train, val, test = self.generate_split(seed)
        train_loader = DataLoader(train, batch_size = batch, shuffle = True)
        val_loader = DataLoader(val, batch_size = batch, shuffle = False)
        test_loader = DataLoader(test, batch_size = batch, shuffle = False)
        return train_loader, val_loader, test_loader
    
datapath = r'data/QM9/processed/qm9_processed.pt'
split = [cfg.train_split, cfg.val_split]     # test = 1 - train - val
seed = cfg.seed; batch = cfg.batch_size
Loader = BuildLoader(datapath, split)
train, val, test = Loader.generate_split(seed = seed)
train_loader, val_loader, test_loader = Loader.load_split(seed = seed, batch = batch)
print("Data loaded successfully.")

Regression works better when the target is normalised; this makes the training process more stable. In order to do so, we first calculate relevant statistics - i.e., mean (mu) and standard deviation (std). 

IMPORTANT: We do this only on the training set to prevent data leakage.

In [ ]:
def training_stats(training_set, target_idx: int):
    ys = []
    for data in training_set: ys.append(data.y[:, target_idx])
    y = torch.cat(ys, dim = 0)
    assert y.numel() > 0, 'Error, please check the target index'
    mu = float(y.mean()); std = float(y.std())
    return y, mu, std

print("Calculating target statistics.")
y, mu, std = training_stats(train, cfg.homo_lumo_idx)
print("target mean:", mu)
print("target std:", std)


def normalise(batch, target_idx, mu, std):
    assert mu is not None and std is not None, "First calculate training stats and then pass mean and std for normalization"
    y = batch.y[:, target_idx]
    return (y - mu) / std

def denormalise(norm, mu, std):
    assert mu is not None and std is not None, "First calculate training stats and then pass mean and std for normalization"
    target = norm * std + mu
    return target    

Data has been sorted, now let's tackle the model. 

## Model construction

In [ ]:
# --- Helper function ---
def segment_sum(src: torch.tensor, index: torch.tensor, num_segments: int):
    '''A grouped summation helper.
    Given a tensor of values that need to be aggregated, say shape: [A];
    index is the segment id for each row in src, has shape [A] and values in [0, num_segments - 1];
    num_segments is the total num of groups
    ---
    We use index_add_ to add each src[i] into out[index[i]]
    '''
    out = torch.zeros(num_segments, src.size(-1), device = src.device, dtype = src.dtype)
    return out.index_add_(0, index, src)

A worked out example:

Say:
```python 
    src = torch.tensor([
        [1.0, 0.5],   # message 0
        [2.0, 1.0],   # message 1
        [0.5, 3.0],   # message 2
        [1.5, 0.5],   # message 3
        [2.0, 2.0],   # message 4
    ])
```
In words, 5 messages, each having 2 features.

AND
```python
    index = torch.tensor([0, 1, 0, 2, 1])  # message i -> segment index[i]
    num_segments = 3
```
In words, which segment/node each message belongs to. 
So the message 0 belongs to segment 0, message 1 to segment 1, message 2 to 0, and so on ...

```python
out = segment_sum(src, index, num_segments)
out should be: tensor([[1.5000, 3.5000], [4.0000, 3.0000], [1.5000, 0.5000]])
    segment 0: [1.0, 0.5] + [0.5, 3.0] = [1.5, 3.5]
    segment 1: [2.0, 1.0] + [2.0, 2.0] = [4.0, 3.0]
    segment 2: [1.5, 0.5]
```

This is effectively a pooling (aggregation) operation over the dataset.

### Building the EGNN

Here comes the heart of the model. Before coding it up, let's take a brief look at the concept.

The EGNN maintains:
- node features `h`
- coordinates `x`

This is to say that as a whole, the EGNN updates the node features and the atomic coordinates.

A key design goal is to **respect geometry**. Say we translate the whole system, we would not expect the behavior to change. The same for rotation; upon rotation scalar features should remain stable.

---

What happens in each layer?

- compute edge messages using node features and pairwise distances. Each edge message input is built by concatenating the source node embedding (size: `hidden_dim`), target node embedding (size: `hidden_dim`), pairwise scalar distance feature (size: `1`) and the edge features (size: `edge_attr_dim`). 
- pass the edge message through a neural network to `message_dim`.

- calculate and update coordinates using relative vectors. We use the squared Euclidean distance as 
$$
rel_{ij} = x_i - x_j
$$
$$
radial_{ij} = ||x_i - x_j||^2
$$
It is called `radial` because it captures distance, not direction. Pay special attention to this part - this is where we **enforce equivariance**.

`coord_mlp(m_ij)` looks at the learned interactions between node `i` and `j` and decides how strongly the edge should influence motion. We bound this interaction using a `tanh` function to prevent a collapse and scale it down using `coord_step_size` to ensure that no edge overly dominates this.  

As an analogy, one can think of this process as deciding whether to 'push' the atoms apart, 'pull' them together or do nothing. 

The weight is a scalar, either positive or negative. We multiply it with the unit direction to turn the edge update into a 3D vector. So `direction` gives orientation and `coord_weight` gives magnitude and sign. 

Thus, the model is **ONLY** allowed to move atoms along relative directions. This is perhaps one of the **key** points here. This preserves **equivariance**. 

Finally, we aggregate all edge updates coming into a destination atom and use it to update coordinates. 

Conceptually, each neighbor proposes a small movement for atom `j`; the model adds those proposals together.

Next, we pool (aggregate) all edge messages that end at the node. Concatenate it to the `hidden_state` and pass it to the `node_mlp`, followed by a normalization. 

---

If one has followed along closely, one question may have popped up: **how does changing coordinates affect prediction performance?** 

It may help, it may not help, it could also do harm. You see, changing coordinates inside the network gives the model more expressive geometric computation. This allows it to iteratively rewrite the geometry. That can help when the target depends on subtle spatial arrangements, local geometric refinement makes downstream interactions easier to represent OR the task is inherently geometric, like denoising, dynamics, or conformation-sensitive scoring. That can hurt when the input geometry is already the thing trusted (which we do for QM9), the target is mostly predictable from fixed geometry plus atom/bond features OR the model learns unphysical coordinate drift that helps training loss but damages generalization. 

By all means, try comparing training performance with `update_coords = True` and `update_coords = False`. 

---

**The key takeaway of this approach is that**:

- distances are invariant
- relative vectors transform equivariantly

That is why the layer respects Euclidean symmetries.


In [ ]:
class EGNNLayer(nn.Module):
    def __init__(
            self, 
            hidden_dim: int,
            edge_attr_dim: int, 
            message_dim: int, 
            residual: bool = True,
            update_coords: bool = True, 
            dropout: float = 0.0,
            coord_step_size: float = 0.1, 
            eps: float = 1e-8,
    ):
        super().__init__()
        self.residual = residual; self.update_coords = update_coords
        self.coord_step_size = coord_step_size; self.eps = eps

        edge_input_dim = 2 * hidden_dim + 1 + edge_attr_dim

        self.edge_mlp = nn.Sequential(
            nn.Linear(edge_input_dim, message_dim),
            nn.SiLU(),
            nn.Linear(message_dim, message_dim),
            nn.SiLU(),
        )

        self.coord_mlp = nn.Sequential(
            nn.Linear(message_dim, message_dim),
            nn.SiLU(),
            nn.Linear(message_dim, 1, bias = False),
        )

        self.node_mlp = nn.Sequential(
            nn.Linear(hidden_dim + message_dim, hidden_dim),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
        )

        self.norm = nn.LayerNorm(hidden_dim)

        # Start coordinate updates near zero
        nn.init.xavier_uniform_(self.coord_mlp[0].weight, gain = 0.5)
        nn.init.zeros_(self.coord_mlp[0].bias)
        nn.init.xavier_uniform_(self.coord_mlp[2].weight, gain = 0.01)

    def forward(self, 
                h: torch.Tensor, 
                x: torch.Tensor,
                edge_index: torch.Tensor, 
                edge_attr: Optional[torch.Tensor] = None):
        
        row, col = edge_index # row is the source node, col the target node
        rel = x[row] - x[col] # enforces equivariance (...)
        radial = (rel ** 2).sum(dim = -1, keepdim = True) # squared Euclidean distance
        
        if edge_attr is None:
            edge_attr = torch.zeros(edge_index.size(1), 0, device = h.device, dtype = h.dtype)

        edge_input = torch.cat([h[row], h[col], radial, edge_attr], dim = -1) 
        # hidden_dim + hidden_dim + 1 + edge_attr_dim 

        m_ij = self.edge_mlp(edge_input) # message sent along the direct edge i -> j
        
        '''WHY? 
        h[row] is the source node features; h[col] is the target node features
        rel is the relative position from j to i'''

        if self.update_coords:
            '''Here we update positions'''
            # Normalize direction so long edges do not dominate the step size.
            dist = torch.sqrt(radial + self.eps)
            # turn this into a unit direction
            direction = rel / dist # direction tells the model which way to move. 

            coord_weight = torch.tanh(self.coord_mlp(m_ij)) * self.coord_step_size
            delta_x_edge = direction * coord_weight
            delta_x = torch.zeros_like(x)
            delta_x.index_add_(0, col, delta_x_edge)
            x = x + delta_x 

        m_i = segment_sum(m_ij, col, h.size(0)) # aggregate all messages that end at each node
        node_input = torch.cat([h, m_i], dim = -1)
        delta_h = self.node_mlp(node_input)

        h = h + delta_h if self.residual else delta_h
        h = self.norm(h)
        return h, x

With a single layer ready, we now stack several EGNN layers and build our regression model. To generate graph embeddings we pool node embeddings. The readout will be a small MLP that predicts one scalar per molecule -- here the HOMO-LUMO gap. 

In [ ]:
class EGNNRegressor(nn.Module):
    def __init__(
        self,
        node_feat_dim: int,
        edge_attr_dim: int,
        hidden_dim: int = 128,
        num_layers: int = 6,
        message_dim: int = 128,
        dropout: float = 0.0,
        coord_step_size: float = 0.1, 
        eps: float = 1e-8,
    ):
        super().__init__()
        
        # --- Node Encoder ---
        self.node_encoder = nn.Sequential(
            nn.Linear(node_feat_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

        # --- EGNN stack ---
        self.egnn_stack = nn.ModuleList([
            EGNNLayer(hidden_dim = hidden_dim, 
                      edge_attr_dim = edge_attr_dim, 
                      message_dim = message_dim,
                      dropout = dropout,
                      coord_step_size = coord_step_size,
                      eps = eps) for _ in range(num_layers)])
        
        # --- EGNN readout ---
        self.readout = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, data):
        h = data.x.float()
        x = data.pos.float()
        edge_index = data.edge_index
        edge_attr = data.edge_attr.float() if data.edge_attr is not None else None
        batch = data.batch

        h = self.node_encoder(h)
        for egnn in self.egnn_stack: h, x = egnn(h, x, edge_index, edge_attr)
        
        '''global_add_pool(h, batch) pools node embeddings into one graph embedding per molecule
        We then map this to the readout -- i.e. the HOMO-LUMO gap.
        '''
        g = global_add_pool(h, batch)
        out = self.readout(g).squeeze(-1)
        return out

In [ ]:
model = EGNNRegressor(
    node_feat_dim = dataset.num_features,
    edge_attr_dim = dataset.num_edge_features,
    hidden_dim = cfg.hidden_dim,
    num_layers = cfg.num_layers,
    message_dim = cfg.message_dim,
    dropout = cfg.dropout,
    coord_step_size = cfg.coord_step_size,
    eps = cfg.eps
).to(cfg.device)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('Model ready to be trained')
print(model)
print(f"Trainable parameters: {num_params:,}")

### The hard part is done, now let's start training the model

We use an AdamW optimizer and use the mean squared error (MSE) loss for training. For evaluation, we use the mean absolute error (MAE). Training is done in batches; while the model itself is not too expensive to train on the **QM9**, training on a GPU will permit larger batch sizes.

**NOTE** that we train the regression model on normalised target values. Evaluation however is done on raw (denormalised values). 

In [ ]:
from torch.optim import AdamW
optim = AdamW(model.parameters(), lr = cfg.lr, weight_decay = cfg.weight_decay)
criterion = nn.MSELoss()

def single_epoch(model, 
              loader, 
              optimizer, 
              criterion, 
              cfg, 
              y_mean, 
              y_std):
    model.train()
    total_loss = 0.0
    total_samples = 0

    for data in tqdm(loader, desc = "train", leave = False):
        data = data.to(cfg.device)

        optimizer.zero_grad()

        pred = model(data)
        target = normalise(data, cfg.homo_lumo_idx, y_mean, y_std)

        loss = criterion(pred, target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        batch_size = target.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

    return total_loss / max(total_samples, 1)

@torch.no_grad()
def evaluate(model, loader, cfg, y_mean, y_std):
    model.eval()

    total_mae = 0.0
    total_samples = 0

    for data in tqdm(loader, desc="eval", leave=False):
        data = data.to(cfg.device)

        pred_norm = model(data)
        target_norm = normalise(data, cfg.homo_lumo_idx, y_mean, y_std)

        pred = denormalise(pred_norm, y_mean, y_std)
        target = denormalise(target_norm, y_mean, y_std)

        total_mae += (pred - target).abs().sum().item()
        total_samples += target.size(0)

    return total_mae / max(total_samples, 1)

In [ ]:
batch = next(iter(train_loader)).to(cfg.device)
out = model(batch)
target = normalise(batch, cfg.homo_lumo_idx, mu, std)

print("Prediction shape:", out.shape)
print("Target shape:", target.shape)
print("Any NaNs in output?", torch.isnan(out).any().item())
print("Any NaNs in target?", torch.isnan(target).any().item())

We save a checkpoint every 10 epochs and keep a running track of the best epoch. Additionally, to prevent overfitting, we have a rolling window that tracks the performance across 8 consecutive epochs. If the performance continuously degrades for 8 epochs, we apply an early stopping criteria. 

In [ ]:
best_val_mae = float("inf")
best_state = None
history = []

patience = 8
min_delta = 1e-5
bad_epochs = 0

ckpt_dir = 'checkpoints'
os.makedirs(ckpt_dir, exist_ok = True)
for epoch in range(1, cfg.epochs + 1):
    train_loss = single_epoch(model, train_loader, optim, criterion, cfg, mu, std)
    val_mae = evaluate(model, val_loader, cfg, mu, std)
    
    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_mae": val_mae,
        }
    )

    print(f"Epoch {epoch:03d} | train_loss={train_loss:.5f} | val_mae={val_mae:.5f}")

    if val_mae < best_val_mae - min_delta:
        best_val_mae = val_mae
        best_state = {k: v.cpu() for k, v in model.state_dict().items()}
        bad_epochs = 0

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optim.state_dict(),
            "best_val_mae": best_val_mae,
            "mu": mu,
            "std": std,
            "history": history,
        },
        os.path.join(ckpt_dir, "best.pt")
    )

    else:
        bad_epochs += 1
    
    if epoch % 10 == 0:
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optim.state_dict(),
            "val_mae": val_mae,
            "mu": mu,
            "std": std,
            "history": history,
        },
        os.path.join(ckpt_dir, f"{epoch}.pt"))
        print(f"Saved a checkpoint: Epoch {epoch}")
    
    if bad_epochs >= patience: print(f'Early stopping initiated'); break
    

### We now have a trained **EGNN** 
The last thing we need to do is evaluate the model on a held-out test set -- i.e., data that the model has never seen before to check whether the performance is generalisable. We calculate the MAE on the test set and also check the $\Delta$ in the MAE between the best epoch on the validation set and the test set. 

In [ ]:
ckpt_dir = 'checkpoints'
ckpt_path = os.path.join(ckpt_dir, 'best.pt')

if not os.path.exists(ckpt_path):
    print(f"No checkpoint found at {ckpt_path}. Run the training cell above first.")
else:
    model = EGNNRegressor(
        node_feat_dim = dataset.num_features,
        edge_attr_dim = dataset.num_edge_features,
        hidden_dim = cfg.hidden_dim,
        num_layers = cfg.num_layers,
        message_dim = cfg.message_dim,
        dropout = cfg.dropout,
        coord_step_size = cfg.coord_step_size,
        eps = cfg.eps,
    ).to(cfg.device)

    checkpoint = torch.load(ckpt_path, map_location = cfg.device)
    model.load_state_dict(checkpoint['model_state_dict'])
    mu = checkpoint['mu']
    std = checkpoint['std']
    val_mae = checkpoint['best_val_mae']
    model.eval()

    print('Model loaded. Running test set evaluation...')
    test_mae = evaluate(model, test_loader, cfg, mu, std)
    print(f'Test MAE: {test_mae}')
    print(f"val_mae - test_mae = {val_mae - test_mae}")

All in all, we have built a solid **E(n)-Equivariant Graph Neural Network** for molecules that can be run end-to-end. Along the way, we also took a peek under the hood to understand how the model works. With this foundation, we can now move toward more complex models. While demonstrated here for molecules, similar approaches can be extended to other types of graph data.